# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Clustering (unsupervised).**

In ML-02 I framed the *decision* as “which pages should the content team fix first?” Clustering does not score pages directly — it groups similar pages so we can see **what kinds of pages exist** in the data (traffic shape, CTR, engagement, rank position) before we build a supervised ranker or classifier in later weeks.

Why clustering here, not classification yet: the starter slice has many numeric signals that interact (median CTR is 0.07%, median impressions 731, median position 11.4 when position is measured). I do not yet know which feature combinations define a “high-opportunity” page. Clustering lets the data suggest natural segments; I will **sense-check** those segments against observed outcomes (e.g. share of declining pages per cluster) without using trend fields as inputs.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**During clustering: no target.** Unsupervised grouping — the algorithm does not predict a label.

**Proxy for later supervised work (not used to fit clusters):** `is_declining_label` from observed impression trend — 1 when `trend_direction == "down"` (~54% of rows in this slice). That label is **observed** from last-30d vs prev-30d impressions, not a hand-written rule I invented. I will **not** put `trend_direction` or `trend_pct` into cluster features (label leakage; see data dictionary).

**How clusters connect to the decision:** after grouping, I compare clusters on observed CTR, engagement, and decline share to name segments (e.g. “high impressions, low CTR, often declining”). That naming step turns exploration into **decision-support** for what to optimize first within each segment.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary (clustering): silhouette score** on a scaled feature matrix — measures how separated and compact clusters are. I will report the value and the chosen *k*; “good” means clusters are interpretable **and** silhouette is not trivially negative.

**Secondary (human sense-check):** for each cluster, inspect median CTR, impressions, position, and the **observed** share with `trend_direction == "down"`. A useful cluster is one where the profile is distinct and actionable for an editor (not just noise).

I am **not** claiming ROC-AUC or precision@K yet — those belong to a later supervised ranking/classification step once a target and baseline queue exist.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import pandas as pd

dataset = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# One row = one content item (page), trailing-90-day metrics for one client assignment
clustering_slice = dataset[
    [
        "content_id",
        "client_id",
        "content_type",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "ctr",
        "engagement_rate",
        "scroll_rate",
        "avg_position",
        "search_volume",
        "content_age_days",
        "days_since_last_update",
        "trend_direction",  # for post-hoc sanity checks only — not a cluster feature
    ]
].copy()

# avg_position = 0 means missing rank data, not rank zero
clustering_slice.loc[clustering_slice["avg_position"] == 0, "avg_position"] = pd.NA

print(f"Rows: {len(clustering_slice):,}  |  Clients: {clustering_slice['client_id'].nunique()}")
print(f"Grain check (one row per page): {clustering_slice['content_id'].is_unique}")
clustering_slice.head()

Rows: 30,000  |  Clients: 32
Grain check (one row per page): True


,content_id,client_id,content_type,impressions_90d,clicks_90d,sessions_90d,ctr,engagement_rate,scroll_rate,avg_position,search_volume,content_age_days,days_since_last_update,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,17,0.76,5.88,4.55,10.6,10.0,187,20,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,9,0.05,0.00,10.00,20.3,90.0,445,25,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,11,0.09,0.00,28.57,36.5,0.0,141,20,down
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,78,0.49,1.28,3.45,6.2,10.0,463,22,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,145,0.13,0.00,24.29,44.0,0.0,263,14,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like “flag pages with impressions > 3,000 and CTR < 0.1%” only catches **one** pattern. This dataset mixes three content types, heavy-tailed traffic (median impressions 731 vs max in the millions), and missing keyword fields on ~2,468 rows — missingness follows `content_type`, so a single threshold rule silently ignores whole groups.

Engagement and search performance also pull in different directions: median CTR is 0.07% while median scroll rate is 5.0%; position and impressions do not line up in one simple if-statement. Clustering on multiple scaled metrics (traffic totals, rates, age, freshness) is a **directional** way to discover combined patterns I would not write by hand — then I prioritize within each cluster using observed outcomes, not product flags.

## Self-check

Before you submit, confirm each line honestly:

- [✔️] Every section above is filled — markdown thinking AND the code that backs it
- [✔️] The notebook runs top to bottom with no errors (Runtime → Run all)
- [] No client names, URLs, or private queries anywhere
- [✔️] My claims use careful words: observed, measured, directional, decision-support
- [✔️] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.